In [41]:
import os

os.chdir(r"C:\Users\HP USER\OneDrive\Desktop\transfer-learning-kaduna-house-price")

print("Current folder:", os.getcwd())
print("Files in data folder:", os.listdir("data"))

Current folder: C:\Users\HP USER\OneDrive\Desktop\transfer-learning-kaduna-house-price
Files in data folder: ['data_kaggle.csv', 'Strict_Kaduna_South_Dataset.csv']


In [42]:
import pandas as pd
import numpy as np

Load Malaysia dataset only

In [43]:
import pandas as pd

malaysia_df = pd.read_csv("data/data_kaggle.csv")

print("Original shape:", malaysia_df.shape)
print(malaysia_df.columns.tolist())

Original shape: (53883, 8)
['Location', 'Price', 'Rooms', 'Bathrooms', 'Car Parks', 'Property Type', 'Size', 'Furnishing']


Select only relevant columns

In [44]:
selected_columns = [
    "Rooms",
    "Size",
    "Bathrooms",
    "Furnishing",
    "Property Type",
    "Price"
]

malaysia_df = malaysia_df[selected_columns]

print("After selecting columns:", malaysia_df.shape)
malaysia_df.head()

After selecting columns: (53883, 6)


,Rooms,Size,Bathrooms,Furnishing,Property Type,Price
0,2+1,"Built-up : 1,335 sq. ft.",3.0,Fully Furnished,Serviced Residence,"RM 1,250,000"
1,6,Land area : 6900 sq. ft.,7.0,Partly Furnished,Bungalow,"RM 6,800,000"
2,3,"Built-up : 1,875 sq. ft.",4.0,Partly Furnished,Condominium (Corner),"RM 1,030,000"
3,NaN,NaN,NaN,NaN,NaN,NaN
4,4+1,"Built-up : 1,513 sq. ft.",3.0,Partly Furnished,Condominium (Corner),"RM 900,000"


# Check missing values before cleaning

In [45]:
# Check missing values before cleaning
print("Shape before missing value handling:", malaysia_df.shape)
print(malaysia_df.isnull().sum())

Shape before missing value handling: (53883, 6)
Rooms            1706
Size             1063
Bathrooms        2013
Furnishing       6930
Property Type      25
Price             248
dtype: int64


# Drop rows with missing values, following the base paper approach

In [46]:
# Drop rows with missing values, following the base paper approach
malaysia_df = malaysia_df.dropna()

print("Shape after dropping missing values:", malaysia_df.shape)
print(malaysia_df.isnull().sum())

Shape after dropping missing values: (45207, 6)
Rooms            0
Size             0
Bathrooms        0
Furnishing       0
Property Type    0
Price            0
dtype: int64


In [47]:
# Check current data types before numeric cleaning
malaysia_df.dtypes

Rooms                str
Size                 str
Bathrooms        float64
Furnishing           str
Property Type        str
Price                str
dtype: object

In [48]:
# Clean Rooms column
malaysia_df["Rooms"] = (
    malaysia_df["Rooms"]
    .astype(str)
    .str.extract(r"(\d+\.?\d*)")
    .astype(float)
)

# Clean Bathrooms column
malaysia_df["Bathrooms"] = (
    malaysia_df["Bathrooms"]
    .astype(str)
    .str.extract(r"(\d+\.?\d*)")
    .astype(float)
)

# Clean Size column
malaysia_df["Size"] = (
    malaysia_df["Size"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.extract(r"(\d+\.?\d*)")
    .astype(float)
)

# Clean Price column
malaysia_df["Price"] = (
    malaysia_df["Price"]
    .astype(str)
    .str.replace("RM", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.extract(r"(\d+\.?\d*)")
    .astype(float)
)

malaysia_df.head()

,Rooms,Size,Bathrooms,Furnishing,Property Type,Price
0,2.0,1335.0,3.0,Fully Furnished,Serviced Residence,1250000.0
1,6.0,6900.0,7.0,Partly Furnished,Bungalow,6800000.0
2,3.0,1875.0,4.0,Partly Furnished,Condominium (Corner),1030000.0
4,4.0,1513.0,3.0,Partly Furnished,Condominium (Corner),900000.0
5,4.0,7200.0,5.0,Partly Furnished,Bungalow,5350000.0


## 5. Checking for Failed Numeric Conversion

After converting the selected columns to numerical format, some values may fail to convert properly.  
Such failed values become missing values again, so they are removed before continuing.

In [49]:
# Check missing values after numeric conversion
malaysia_df.isnull().sum()

Rooms            737
Size              46
Bathrooms          0
Furnishing         0
Property Type      0
Price              0
dtype: int64

In [50]:
# Drop rows where numeric conversion failed
malaysia_df = malaysia_df.dropna()

print("Shape after numeric cleaning:", malaysia_df.shape)
malaysia_df.head()

Shape after numeric cleaning: (44425, 6)


,Rooms,Size,Bathrooms,Furnishing,Property Type,Price
0,2.0,1335.0,3.0,Fully Furnished,Serviced Residence,1250000.0
1,6.0,6900.0,7.0,Partly Furnished,Bungalow,6800000.0
2,3.0,1875.0,4.0,Partly Furnished,Condominium (Corner),1030000.0
4,4.0,1513.0,3.0,Partly Furnished,Condominium (Corner),900000.0
5,4.0,7200.0,5.0,Partly Furnished,Bungalow,5350000.0


In [51]:
original_cols = [
    "Rooms",
    "Size",
    "Bathrooms",
    "Furnishing",
    "Property Type",
    "Price"
]

print("Duplicates before conversion:",
      malaysia_df.duplicated(subset=original_cols).sum())

Duplicates before conversion: 6093


## Removing duplicates based on original Malaysia dataset features

In [52]:
original_cols = [
    "Rooms",
    "Size",
    "Bathrooms",
    "Furnishing",
    "Property Type",
    "Price"
]

malaysia_df = malaysia_df.drop_duplicates(subset=original_cols)

print("Shape after removing duplicates correctly:", malaysia_df.shape)

Shape after removing duplicates correctly: (38332, 6)


In [53]:
# Remove duplicate rows
malaysia_df = malaysia_df.drop_duplicates()

print("Shape after removing duplicates:", malaysia_df.shape)

Shape after removing duplicates: (38332, 6)


## 6. Converting Malaysia Price from Ringgit to Naira

The Malaysia dataset price is originally in Malaysian Ringgit (MYR).  
Since the Kaduna dataset is in Nigerian Naira, the Malaysia price is converted to Naira before alignment.

In [54]:
MYR_TO_NGN = 350

malaysia_df["Price_Naira"] = malaysia_df["Price"] * MYR_TO_NGN

malaysia_df[["Price", "Price_Naira"]].head()

,Price,Price_Naira
0,1250000.0,4.375000e+08
1,6800000.0,2.380000e+09
2,1030000.0,3.605000e+08
4,900000.0,3.150000e+08
5,5350000.0,1.872500e+09


In [55]:
# Remove duplicate rows
malaysia_df = malaysia_df.drop_duplicates()

print("Shape after removing duplicates:", malaysia_df.shape)

Shape after removing duplicates: (38332, 7)


In [56]:
print(malaysia_df.columns)

Index(['Rooms', 'Size', 'Bathrooms', 'Furnishing', 'Property Type', 'Price',
       'Price_Naira'],
      dtype='str')


In [57]:
malaysia_df.shape

(38332, 7)

Use only one price column

In [59]:
malaysia_df = malaysia_df.drop(columns=["Price"])
malaysia_df = malaysia_df.rename(columns={"Price_Naira": "Price"})

In [60]:
print(malaysia_df.columns)

Index(['Rooms', 'Size', 'Bathrooms', 'Furnishing', 'Property Type', 'Price'], dtype='str')


Confirm no duplicates remain

In [61]:
original_cols = [
    "Rooms",
    "Size",
    "Bathrooms",
    "Furnishing",
    "Property Type",
    "Price"
]

print("Remaining duplicates:",
      malaysia_df.duplicated(subset=original_cols).sum())

Remaining duplicates: 0


Confirm no missing values

In [62]:
print(malaysia_df.isnull().sum())

Rooms            0
Size             0
Bathrooms        0
Furnishing       0
Property Type    0
Price            0
dtype: int64


Confirm correct data types

In [63]:
print(malaysia_df.dtypes)

Rooms            float64
Size             float64
Bathrooms        float64
Furnishing           str
Property Type        str
Price            float64
dtype: object


Check for unrealistic values

In [64]:
malaysia_df.describe()

,Rooms,Size,Bathrooms,Price
count,38332.000000,38332.000000,38332.000000,3.833200e+04
mean,3.312089,2231.160922,3.171319,6.542298e+08
std,1.331076,9398.381833,1.667441,2.985979e+09
min,1.000000,0.000000,1.000000,1.078000e+05
25%,3.000000,915.000000,2.000000,2.100000e+08
50%,3.000000,1290.000000,3.000000,3.675000e+08
75%,4.000000,2207.000000,4.000000,7.280000e+08
max,20.000000,820000.000000,20.000000,5.600000e+11
